# s_spatioloji: Basic Workflow

This notebook demonstrates the core data processing pipeline for image-based spatial transcriptomics using `s_spatioloji`.

**What you'll learn:**
- Loading data from Xenium (or MERSCOPE) output
- Inspecting the dataset (cells, genes, images)
- Normalization, feature selection, and scaling
- Dimensionality reduction (PCA, UMAP)
- Clustering (Leiden)
- Basic visualization (embedding scatter, heatmap, violin, dotplot)

## 1. Load Data

Load a Xenium output directory into the standardized `s_spatioloji` format.
This creates a processed dataset with expression (Zarr), cell metadata (Parquet),
boundaries (GeoParquet), and images (OME-TIFF).

In [ ]:
from pathlib import Path
from s_spatioloji.data.io import from_xenium
from s_spatioloji.data.core import s_spatioloji

# --- Option A: Load from raw Xenium output ---
# sj = from_xenium(
#     xenium_dir="/path/to/xenium_output",
#     output_dir="/path/to/processed_dataset",
#     use_nucleus_boundaries=False,  # use cell boundaries (default)
#     copy_image=False,              # symlink images to save disk space
# )

# --- Option B: Open an already-processed dataset ---
sj = s_spatioloji.open("/path/to/processed_dataset")

print(sj)

## 2. Inspect the Dataset

All backends are lazy-loaded. Metadata queries (cell count, gene count) read only file headers, not full data.

In [ ]:
# Basic stats
print(f"Cells:  {sj.n_cells:,}")
print(f"Genes:  {sj.n_genes:,}")
print(f"Boundaries: {sj.has_boundaries}")
print(f"Images: {sj.has_images}")

# Cell metadata columns
print(f"\nCell columns: {sj.cells.columns}")

In [ ]:
# Preview cell metadata (lazy dask DataFrame -> compute to pandas)
cells_df = sj.cells.df.head(10).compute()
cells_df

In [ ]:
# Gene names
gene_names = sj.expression.gene_names
print(f"First 10 genes: {gene_names[:10]}")
print(f"Last 10 genes:  {gene_names[-10:]}")

In [ ]:
# If images are available, inspect the collection
if sj.has_images:
    print(f"Available images: {sj.images.keys()}")
    print(f"Pixel size (level 0): {sj.images.pixel_size} um/px")
    print(f"Pixel size (level 3): {sj.images.pixel_size_at(3)} um/px")
    print(f"Default image: {sj.images.default_image}")

## 3. Normalization Pipeline

Standard single-cell processing: total-count normalization, log1p transform,
highly variable gene selection, and scaling.

All results are persisted to `maps/` as Parquet or Zarr. Re-running with `force=False` skips already-computed steps.

In [ ]:
from s_spatioloji.compute.normalize import normalize_total, log1p, scale
from s_spatioloji.compute.feature_selection import highly_variable_genes

# Step 1: Normalize to 10,000 counts per cell
normalize_total(sj, target_sum=1e4)

# Step 2: Log1p transform
log1p(sj)

# Step 3: Select highly variable genes
highly_variable_genes(sj, n_top=2000)

# Step 4: Scale (z-score, clipped to max_value=10)
scale(sj, hvg=True)

print("Normalization pipeline complete.")
print(f"Available maps: {sj.maps.keys()}")

## 4. Dimensionality Reduction

PCA on scaled HVGs, then UMAP for visualization.

In [ ]:
from s_spatioloji.compute.reduction import pca, umap

# PCA: 50 components on scaled HVGs
pca(sj, n_components=50)

# UMAP: 2D embedding from PCA
umap(sj, n_neighbors=15)

print("Dimensionality reduction complete.")

## 5. Clustering

Leiden community detection on a KNN graph built from PCA coordinates.

In [ ]:
from s_spatioloji.compute.clustering import leiden

leiden(sj, resolution=1.0, n_neighbors=15)

# Preview cluster assignments
clusters = sj.maps["leiden"].compute()
print(f"Number of clusters: {clusters['leiden'].nunique()}")
print(clusters['leiden'].value_counts().head(10))

## 6. Visualization

All plotting functions return `matplotlib.axes.Axes` and auto-save to `{dataset}/figures/`.

In [ ]:
import matplotlib.pyplot as plt
from s_spatioloji.visualization.embedding import scatter

# UMAP colored by Leiden clusters
scatter(sj, basis="X_umap", color_by="leiden", point_size=1, figsize=(10, 8))
plt.show()

In [ ]:
# UMAP colored by a gene
scatter(sj, basis="X_umap", color_by="gene_0", cmap="magma", point_size=1, figsize=(10, 8))
plt.show()

In [ ]:
from s_spatioloji.visualization.expression import heatmap, violin, dotplot

# Select marker genes (replace with your actual markers)
markers = gene_names[:10]  # first 10 genes as placeholder

# Heatmap: mean expression per cluster (z-scored)
heatmap(sj, genes=markers, cluster_key="leiden", standardize=True)
plt.show()

In [ ]:
# Violin plot: expression distribution per cluster
fig = violin(sj, genes=markers[:4], cluster_key="leiden", ncols=2)
plt.show()

In [ ]:
# Dot plot: fraction expressing + mean expression
dotplot(sj, genes=markers, cluster_key="leiden")
plt.show()

## 7. Spatial Visualization

Plot cells at their physical (x, y) coordinates.

In [ ]:
from s_spatioloji.visualization.spatial import spatial_scatter, spatial_expression

# Spatial map colored by cluster
spatial_scatter(sj, color_by="leiden", point_size=0.5, figsize=(12, 12))
plt.show()

In [ ]:
# Spatial gene expression overlay
spatial_expression(sj, gene=markers[0], cmap="magma", point_size=0.5, figsize=(12, 12))
plt.show()

In [ ]:
# Zoom into a region of interest
spatial_scatter(
    sj, color_by="leiden",
    xlim=(0, 2000), ylim=(0, 2000),  # crop to 2mm x 2mm
    point_size=3, figsize=(10, 10),
)
plt.show()

In [ ]:
# Overlay on tissue image (if available)
# spatial_scatter(
#     sj, color_by="leiden",
#     image_path=sj.images["morphology_focus_0000"].path,
#     image_alpha=0.4, point_size=0.3,
#     figsize=(14, 14),
# )
# plt.show()

## 8. Batch Correction (Optional)

If your dataset has multiple FOVs with batch effects, apply Harmony on PCA coordinates.

In [ ]:
# from s_spatioloji.compute.batch_correction import harmony
# from s_spatioloji.compute.reduction import umap as run_umap
#
# # Harmony on PCA
# harmony(sj, batch_key="fov_id")
#
# # Re-run UMAP on corrected PCA
# run_umap(sj, input_key="X_pca_harmony", output_key="X_umap_harmony")
#
# # Compare before/after
# fig, axes = plt.subplots(1, 2, figsize=(18, 7))
# scatter(sj, basis="X_umap", color_by="fov_id", ax=axes[0], save=False)
# axes[0].set_title("Before Harmony")
# scatter(sj, basis="X_umap_harmony", color_by="fov_id", ax=axes[1], save=False)
# axes[1].set_title("After Harmony")
# plt.tight_layout()
# plt.show()

## Summary

This notebook demonstrated the core `s_spatioloji` workflow:

1. **Load** data from Xenium/MERSCOPE output
2. **Normalize** (total counts, log1p, HVG selection, scaling)
3. **Reduce** dimensionality (PCA, UMAP)
4. **Cluster** cells (Leiden)
5. **Visualize** embeddings, expression, and spatial maps

All computed results are persisted to `maps/` for downstream analysis. See `02_spatial_analysis.ipynb` for spatial statistics and pattern analysis.